In [2]:
import pickle
import numpy as np
import dunex_paths
from glob import glob 
import os

In [10]:
files = glob('../data/argus/metadata/*.pickle')
print(files)
total_seconds = 0
for file_path in files:
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
        
        active_indices = np.where(data['captureArray'] > 0)[0]
        
        if active_indices.size > 0:
            first_idx = active_indices[0]
            last_idx = active_indices[-1]
            
            # This creates a timedelta object
            diff = data['time'][last_idx] - data['time'][first_idx]
            
            # Convert to a float of total seconds
            duration_seconds = diff.total_seconds()
            
            total_seconds += duration_seconds
            fname = os.path.basename(file_path)
            print(f"{fname:<40} | {duration_seconds:<12.2f}")
        else:
            print(f"{os.path.basename(file_path):<40} | No captures")
            
    except Exception as e:
        print(f"Error processing {os.path.basename(file_path)}: {e}")
        
        
print(total_seconds / 60 / 60, 'H')

['../data/argus/metadata/ArgusFF_20210919T203000Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20211008T130100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20210918T130000Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20211010T150100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20210929T130100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20211013T153100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20211026T130100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20210929T190100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20210918T190000Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20210921T130100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20211028T123100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20210924T120100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20211027T113100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_20211006T123100Z_headerData.pickle', '../data/argus/metadata/ArgusFF_2

In [12]:
import glob
import os
import pickle

import numpy as np

video_dir = dunex_paths.ARGUS_VIDEO_DIR
metadata_dir = dunex_paths.DATA_ROOT / 'argus' / 'metadata'

# 1. Get all video files and extract their timestamps
# Pattern: ArgusFF_[TIMESTAMP]_RectifiedVideo.avi
video_files = glob.glob(os.path.join(video_dir, '*.avi'))
video_timestamps = [os.path.basename(f).split('_')[1] for f in video_files]

print(f"Found {len(video_timestamps)} videos in dataset. Matching with metadata...\n")
print(f"{'Timestamp':<20} | {'Duration (s)':<12}")
print("-" * 35)

total_subset_duration = 0.0

# 2. Iterate through timestamps and find the corresponding pickle
for ts in sorted(video_timestamps):
    # Construct the expected pickle filename pattern
    pickle_glob = os.path.join(metadata_dir, f"ArgusFF_{ts}_headerData.pickle")
    match = glob.glob(pickle_glob)
    
    if match:
        file_path = match[0]
        try:
            with open(file_path, 'rb') as f:
                data = pickle.load(f)
            
            active_indices = np.where(data['captureArray'] > 0)[0]
            
            if active_indices.size > 0:
                diff = data['time'][active_indices[-1]] - data['time'][active_indices[0]]
                duration_seconds = diff.total_seconds()
                
                total_subset_duration += duration_seconds
                print(f"{ts:<20} | {duration_seconds:<12.2f}")
            else:
                print(f"{ts:<20} | No active captures")
                
        except Exception as e:
            print(f"{ts:<20} | Error: {e}")
    else:
        print(f"{ts:<20} | Metadata not found")

print("-" * 35)
print(f"{'TOTAL SUBSET':<20} | {total_subset_duration:<12.2f} seconds")
print(f"{'TOTAL (Hours)':<20} | {total_subset_duration/3600:<12.2f} hours")

Found 36 videos in dataset. Matching with metadata...

Timestamp            | Duration (s)
-----------------------------------
20210917T143000Z     | 1199.00     
20210917T163000Z     | 1199.00     
20210917T173000Z     | 1199.50     
20210918T120000Z     | 1199.00     
20210918T130000Z     | 1199.00     
20210919T153000Z     | 1199.00     
20210919T163000Z     | 1199.00     
20210919T180000Z     | 1199.00     
20210919T183000Z     | 1199.00     
20210920T190100Z     | 936.50      
20210921T120100Z     | 1199.50     
20210926T120100Z     | 1199.50     
20210926T160100Z     | 1199.50     
20210927T113100Z     | 1199.50     
20210927T130100Z     | 1199.50     
20210927T170100Z     | 651.00      
20210928T213100Z     | 1199.50     
20210929T120100Z     | 1570.00     
20210929T183100Z     | 1511.50     
20210929T200100Z     | 1563.00     
20211006T120100Z     | 1408.50     
20211006T193100Z     | 1644.00     
20211007T133100Z     | 1490.00     
20211010T150100Z     | 1280.50     
20211010T